# Humor Vibes Open — starter notebook (Track A)

Pure-stdlib starter: no pandas, no torch, no internet required. It

1. finds the competition data (Kaggle input or a local checkout),
2. loads `train.csv` / `test.csv` with the `csv` module and prints quick EDA counts,
3. scores a deliberately trivial punctuation/length/overlap heuristic,
4. writes a **valid** `submission.csv` (`id,humor_score` for every test id),
5. self-scores with the competition's own `metric_humor_vibes.py` on the labeled
   train split (and on `solution.csv` when run host-side).

**Task**: each test item is a (`setup`, `punchline`) pair. Submit a `humor_score`
per id — higher = more likely a genuine human joke, lower = constructed control
(a shuffled punchline from a different setup, or a deliberately boring tail).
The metric is rank-based AUC, so only the *ordering* of your scores matters.

The intended real baseline is Gemma-as-instrument (teacher-forced surprisal +
null control) — see the competition description. This notebook is the floor,
not the ceiling.

In [ ]:
import csv, glob, os
from pathlib import Path

def find_data_dir():
    hits = glob.glob('/kaggle/input/*/test.csv') + glob.glob('/kaggle/input/*/*/test.csv')
    if hits:
        return Path(sorted(hits)[0]).parent
    for cand in ('competition/data', 'data', '../data', '../competition/data'):
        if (Path(cand) / 'test.csv').exists():
            return Path(cand)
    raise FileNotFoundError('test.csv not found — attach the competition data or run from the repo root')

def find_metric():
    hits = glob.glob('/kaggle/input/**/metric_humor_vibes.py', recursive=True)
    if hits:
        return Path(sorted(hits)[0])
    for cand in ('competition/metric_humor_vibes.py', 'metric_humor_vibes.py',
                 '../metric_humor_vibes.py', '../../metric_humor_vibes.py'):
        if Path(cand).exists():
            return Path(cand)
    return DATA_DIR.parent / 'metric_humor_vibes.py'   # repo layout: data/ sits beside the metric

DATA_DIR = find_data_dir()
METRIC_PATH = find_metric()
# submission lands in /kaggle/working on Kaggle, cwd locally; HV_OUT overrides
OUT_DIR = Path(os.environ.get('HV_OUT') or ('/kaggle/working' if Path('/kaggle/working').exists() else '.'))
print('data dir  :', DATA_DIR.resolve())
print('metric    :', METRIC_PATH.resolve() if METRIC_PATH.exists() else '(not found — self-scoring cells will skip)')
print('output dir:', OUT_DIR.resolve())

In [ ]:
def load_csv(name):
    with (DATA_DIR / name).open(newline='', encoding='utf-8') as fh:
        return list(csv.DictReader(fh))

train = load_csv('train.csv')
test = load_csv('test.csv')
print(f'train: {len(train)} rows | test: {len(test)} rows')
genuine = [r for r in train if r['is_genuine'] == '1']
controls = [r for r in train if r['is_genuine'] == '0']
print(f'train genuine: {len(genuine)} | controls: {len(controls)}')
by_type = {}
for r in controls:
    by_type[r['control_type']] = by_type.get(r['control_type'], 0) + 1
print('control types:', by_type)
for label, rows in (('genuine', genuine), ('control', controls)):
    lens = [len(r['punchline'].split()) for r in rows]
    print(f'{label} punchline words: mean {sum(lens)/len(lens):.1f}, min {min(lens)}, max {max(lens)}')
print('\nsample genuine :', genuine[0]['setup'], '=>', genuine[0]['punchline'])
print('sample control :', controls[0]['setup'], '=>', controls[0]['punchline'],
      f"[{controls[0]['control_type']}]")

## A trivial baseline: punctuation, length, and setup-echo
No model, no training — three surface signals:

- **punctuation/casing**: authored punchlines tend to open capitalized and carry
  punch marks (`!`, quotes); boring continuation tails read like mid-sentence prose;
- **setup echo**: the boring tails re-use a word from the setup by construction, so
  content-word overlap with the setup is (weak) evidence *against* a joke;
- **length**: punchlines cluster short; very long tails are suspicious.

This mostly separates *boring* controls and is near-chance on *shuffled* ones —
shuffled punchlines are real punchlines attached to the wrong setup, and telling
them apart needs actual setup-punchline coherence (e.g. Gemma logits). That gap is
the competition.

In [ ]:
STRIP = '.,!?\"\'()[]:;'

def content_words(text):
    return {w.strip(STRIP).lower() for w in text.split() if len(w.strip(STRIP)) > 3}

def baseline_score(setup, punchline):
    p = punchline.strip()
    words = p.split()
    score = 0.0
    if p[:1].isupper():
        score += 1.0                                  # authored-punchline casing
    score += 0.6 * p.count('!') + 0.3 * p.count('"')  # punch marks
    score -= 0.8 * len(content_words(setup) & content_words(p))   # setup echo
    score -= 0.05 * abs(len(words) - 8)               # mild length prior
    return round(score, 4)

preds = {r['id']: baseline_score(r['setup'], r['punchline']) for r in test}
print('scored', len(preds), 'test items | score range:',
      f"{min(preds.values()):.2f} .. {max(preds.values()):.2f}")

In [ ]:
sub_path = OUT_DIR / 'submission.csv'
with sub_path.open('w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh)
    w.writerow(['id', 'humor_score'])
    for r in test:                       # keep test.csv order; every id exactly once
        w.writerow([r['id'], preds[r['id']]])
print('wrote', sub_path, f'({len(test)} rows + header)')

## Self-scoring with the competition metric
`metric_humor_vibes.py` ships in the data bundle — it is the canonical scorer
(dependency-free, `score(solution, submission, "id")`). The train split has
public labels, so you can iterate locally without burning submissions. When the
host runs this notebook next to `solution.csv`, the same cell reports the real
test AUC plus the control diagnostics.

In [ ]:
import importlib.util

metric = None
if METRIC_PATH.exists():
    spec = importlib.util.spec_from_file_location('metric_humor_vibes', str(METRIC_PATH))
    metric = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(metric)
else:
    print('metric_humor_vibes.py not found — skipping self-scoring')

if metric:
    train_sub = [{'id': r['id'], 'humor_score': baseline_score(r['setup'], r['punchline'])}
                 for r in train]
    train_auc = metric.score(train, train_sub, 'id')
    print(f'baseline AUC on labeled train split: {train_auc}')

In [ ]:
# Host-side extras: with solution.csv present this reports the real test AUC
# and the metric's control readouts. Participants: this cell just skips.
sol_path = DATA_DIR / 'solution.csv'
if metric and sol_path.exists():
    with sol_path.open(newline='', encoding='utf-8') as fh:
        sol = list(csv.DictReader(fh))
    test_sub = [{'id': r['id'], 'humor_score': preds[r['id']]} for r in test]
    print('TEST baseline AUC (all controls)   :', metric.score(sol, test_sub, 'id'))
    for ctype in ('shuffled', 'boring'):
        part = [r for r in sol if r['is_genuine'] == '1' or r['control_type'] == ctype]
        print(f'TEST AUC vs {ctype:8s} controls only :', metric.score(part, test_sub, 'id'))
    for usage in ('Public', 'Private'):
        part = [r for r in sol if r['Usage'] == usage]
        print(f'TEST AUC {usage:7s} split            :', metric.score(part, test_sub, 'id'))
    print('matched-pair accuracy (genuine vs shuffled, same setup):',
          metric.matched_pair_accuracy(sol, test_sub, 'id'))
else:
    print('solution.csv not available here (as expected for participants) — skipped')

## Sanity: the metric resists trivial submissions
A constant submission ties every item (AUC 0.5 by rank-average); a random
shuffle of the baseline's own scores lands near 0.5. Seeded, so the numbers
reproduce.

In [ ]:
import random

if metric:
    ids = [r['id'] for r in train]
    const_sub = [{'id': i, 'humor_score': 0.5} for i in ids]
    vals = [baseline_score(r['setup'], r['punchline']) for r in train]
    random.Random(7).shuffle(vals)
    shuf_sub = [{'id': i, 'humor_score': v} for i, v in zip(ids, vals)]
    print('train AUC, all-constant submission :', metric.score(train, const_sub, 'id'))
    print('train AUC, seed-7 shuffled scores  :', metric.score(train, shuf_sub, 'id'))
    if sol_path.exists():
        tids = [r['id'] for r in test]
        tvals = [preds[i] for i in tids]
        random.Random(7).shuffle(tvals)
        print('TEST AUC, all-constant submission  :',
              metric.score(sol, [{'id': i, 'humor_score': 0.5} for i in tids], 'id'))
        print('TEST AUC, seed-7 shuffled scores   :',
              metric.score(sol, [{'id': i, 'humor_score': v} for i, v in zip(tids, tvals)], 'id'))

## Where to go from here
1. **Gemma as the instrument** (the intended baseline): teacher-forced surprisal
   of the punchline given the setup, minus a null-control reading — shuffled
   punchlines are surprising *without* a cheap re-route, genuine ones resolve.
2. **Setup-punchline coherence**: any embedding similarity beats bag-of-words
   overlap on the shuffled controls.
3. **Don't chase the boring tails**: they are template-varied with setup-derived
   words on purpose; a regex list will not generalize (the hosts adversarially
   checked). Model the humor, not the artifact.

Rules note: only the *ordering* of `humor_score` matters; every test id must
appear exactly once, scores must be finite numbers.